In [1]:
# -*- coding: utf-8 -*-
"""
IDBS_Config_test.py —— 新版 AB 代码（IDBS_PlanA.cpp / IDBS_PlanB.cpp 及其 OpenMP 版）的 Python 调用示例

编译动态库（Linux）：
    g++ -std=c++17 -O2 -fopenmp -shared -fPIC IDBS_PlanA_OpenMP.cpp -o jinjie_libtetris.so
    g++ -std=c++17 -O2 -shared -fPIC IDBS_PlanA.cpp -o jinjie_libtetris.so      # 串行版可去掉 -fopenmp
（Windows 把 -shared -fPIC 换成 -shared 即可生成 .dll）

新接口（IDBS_Config）与旧接口（IDBS）的区别：
    旧接口 IDBS(blocks[7], orders[7]) 使用代码内置的默认坐标；
    新接口 IDBS_Config(blocks[7], orders[7], block_coord[70], grid_coord[280], shooting_pose[3])
    把 方块数量 / 放置顺序 / 方块抓取坐标 / 托盘格点坐标 / 初始拍摄姿态 全部由调用方传入，
    运行时不再依赖写死的坐标。

输出格式（每个方块 5 个字段 + 末尾 1 个满行数）：
    名称,角度,x,y,方块索引, 名称,角度,x,y,方块索引, ... ,满行数
    方块索引 = 该实例在传入的 block_coord[方块id] 世界坐标表中的下标（0 起），
    即"选定的方块"对应的抓取坐标下标。
"""

'\nIDBS_Config_test.py —— 新版 AB 代码（IDBS_PlanA.cpp / IDBS_PlanB.cpp 及其 OpenMP 版）的 Python 调用示例\n\n编译动态库（Linux）：\n    g++ -std=c++17 -O2 -fopenmp -shared -fPIC IDBS_PlanA_OpenMP.cpp -o jinjie_libtetris.so\n    g++ -std=c++17 -O2 -shared -fPIC IDBS_PlanA.cpp -o jinjie_libtetris.so      # 串行版可去掉 -fopenmp\n（Windows 把 -shared -fPIC 换成 -shared 即可生成 .dll）\n\n新接口（IDBS_Config）与旧接口（IDBS）的区别：\n    旧接口 IDBS(blocks[7], orders[7]) 使用代码内置的默认坐标；\n    新接口 IDBS_Config(blocks[7], orders[7], block_coord[70], grid_coord[280], shooting_pose[3])\n    把 方块数量 / 放置顺序 / 方块抓取坐标 / 托盘格点坐标 / 初始拍摄姿态 全部由调用方传入，\n    运行时不再依赖写死的坐标。\n\n输出格式（每个方块 5 个字段 + 末尾 1 个满行数）：\n    名称,角度,x,y,方块索引, 名称,角度,x,y,方块索引, ... ,满行数\n    方块索引 = 该实例在传入的 block_coord[方块id] 世界坐标表中的下标（0 起），\n    即"选定的方块"对应的抓取坐标下标。\n'

In [2]:
from ctypes import *
import os

if os.name == "nt":
    # Windows 下 Python 3.8+ 不沿 PATH 搜索动态库依赖（libgomp/libstdc++ 等），
    # 找到 MinGW 运行库目录并加入 DLL 搜索路径（Linux 下无此问题）
    for d in os.environ.get("PATH", "").split(os.pathsep):
        if os.path.exists(os.path.join(d, "libgomp-1.dll")):
            os.add_dll_directory(d)
            break

# 加载动态库（默认是你的 Linux 路径；也可通过环境变量 IDBS_LIB 指定，便于测试）
test = CDLL(os.environ.get("IDBS_LIB", "/home/czx/catkin_ws/src/机械臂资料/带距离进阶任务/IDBSA.so"))

test.IDBS_Config.restype = c_char_p
test.IDBS_Config.argtypes = [POINTER(c_int), POINTER(c_int),
                              POINTER(c_double), POINTER(c_double), POINTER(c_double)]
# 旧接口（内置默认坐标）如需使用：
# test.IDBS.restype = c_char_p
# test.IDBS.argtypes = [POINTER(c_int), POINTER(c_int)]

In [3]:
def flatten(coords):
    """把坐标表展平成 ctypes 需要的连续数组。
    兼容两种写法：[[x,y],[x,y],...]（block_coord 的扁平式）和
    [[[x,y],...],...]（grid_coord 的嵌套式）。"""
    out = []
    for row in coords:
        if isinstance(row[0], (list, tuple)):
            for p in row:
                out.append(p[0])
                out.append(p[1])
        else:
            out.append(row[0])
            out.append(row[1])
    return out


def get_put_table(cube_count, place_order):
    # 把参数打包成 ctypes 数组
    blocks_arr = (c_int * 7)(*cube_count)
    orders_arr = (c_int * 7)(*place_order)
    bc_arr = (c_double * 70)(*flatten(block_coord))
    gc_arr = (c_double * 280)(*flatten(grid_coord))
    pose_arr = (c_double * 3)(*shooting_pose)

    # 调用新接口：数量/顺序/抓取坐标/格点坐标/拍摄姿态全部由调用方传入
    result = test.IDBS_Config(blocks_arr, orders_arr, bc_arr, gc_arr, pose_arr)
    result = result.decode('gbk')

    cube_list0 = result.split(',')
    length = len(cube_list0) // 5   # 每条 5 个字段：名称,角度,x,y,方块索引
    cube_list = []
    for i in range(length):
        cube_list.append([])
        cube_list[i].append(float(cube_list0[i * 5 + 2]) + 0.5)   # x
        cube_list[i].append(float(cube_list0[i * 5 + 3]) + 0.5)   # y
        cube_list[i].append(float(cube_list0[i * 5 + 1]))         # 角度
        cube_name = cube_list0[i * 5]
        if cube_name == "Line":
            cube_name = 'line'
        cube_list[i].append(cube_name)
        cube_list[i].append(int(cube_list0[i * 5 + 4]))           # 方块索引（block_coord 中的下标）

    cube_sum = 0
    for i in cube_count:
        cube_sum = cube_sum + i * 4
    level = cube_sum // 10
    print("极限填满行", level)
    return cube_list, cube_list0[-1]


In [6]:
# ================= 方块数量与放置顺序（与旧版 Python 示例一致）=================
cube_count = [4, 5, 4, 4, 5, 4, 4]   # LR LL ZL ZR O T Line
place_order = [2, 6, 5, 0, 3, 1, 4]

# ================= 初始拍摄姿态（与 C++ 内置默认一致）=================
shooting_pose = [0.0, 0.0, 0.0]

# ================= 方块抓取坐标 blockCoord[7][5][2]（与 C++ 写死的默认坐标一致）=================
block_coord = [
    # 方块0 LR
    [0.912, 2.082], [1.362, 2.318], [1.094, 1.926], [1.224, 2.468], [0.000, 0.000],
    # 方块1 LL
    [0.936, 2.276], [1.344, 2.108], [1.186, 1.918], [1.056, 2.486], [0.884, 2.352],
    # 方块2 ZL
    [1.338, 2.264], [0.918, 2.164], [1.168, 1.932], [1.074, 2.476], [0.000, 0.000],
    # 方块3 ZR
    [0.928, 2.342], [1.356, 2.196], [1.246, 1.938], [1.024, 2.458], [0.000, 0.000],
    # 方块4 O
    [1.326, 2.074], [0.902, 2.226], [1.214, 1.922], [1.088, 2.492], [1.354, 2.382],
    # 方块5 T
    [0.946, 2.044], [1.332, 2.328], [1.142, 1.914], [1.252, 2.462], [0.000, 0.000],
    # 方块6 Line
    [0.896, 2.138], [1.348, 2.176], [1.202, 1.934], [1.042, 2.448], [0.000, 0.000],
]

# ================= 托盘格点坐标 gridCoord[14][10][2]（与 C++ 写死的默认坐标一致）=================
grid_coord = [
    # 第0行 (y=2.000)
    [1.000, 2.000], [1.030, 2.000], [1.060, 2.000], [1.090, 2.000], [1.120, 2.000],
    [1.150, 2.000], [1.180, 2.000], [1.210, 2.000], [1.240, 2.000], [1.270, 2.000],
    # 第1行 (y=2.030)
    [1.000, 2.030], [1.030, 2.030], [1.060, 2.030], [1.090, 2.030], [1.120, 2.030],
    [1.150, 2.030], [1.180, 2.030], [1.210, 2.030], [1.240, 2.030], [1.270, 2.030],
    # 第2行 (y=2.060)
    [1.000, 2.060], [1.030, 2.060], [1.060, 2.060], [1.090, 2.060], [1.120, 2.060],
    [1.150, 2.060], [1.180, 2.060], [1.210, 2.060], [1.240, 2.060], [1.270, 2.060],
    # 第3行 (y=2.090)
    [1.000, 2.090], [1.030, 2.090], [1.060, 2.090], [1.090, 2.090], [1.120, 2.090],
    [1.150, 2.090], [1.180, 2.090], [1.210, 2.090], [1.240, 2.090], [1.270, 2.090],
    # 第4行 (y=2.120)
    [1.000, 2.120], [1.030, 2.120], [1.060, 2.120], [1.090, 2.120], [1.120, 2.120],
    [1.150, 2.120], [1.180, 2.120], [1.210, 2.120], [1.240, 2.120], [1.270, 2.120],
    # 第5行 (y=2.150)
    [1.000, 2.150], [1.030, 2.150], [1.060, 2.150], [1.090, 2.150], [1.120, 2.150],
    [1.150, 2.150], [1.180, 2.150], [1.210, 2.150], [1.240, 2.150], [1.270, 2.150],
    # 第6行 (y=2.180)
    [1.000, 2.180], [1.030, 2.180], [1.060, 2.180], [1.090, 2.180], [1.120, 2.180],
    [1.150, 2.180], [1.180, 2.180], [1.210, 2.180], [1.240, 2.180], [1.270, 2.180],
    # 第7行 (y=2.210)
    [1.000, 2.210], [1.030, 2.210], [1.060, 2.210], [1.090, 2.210], [1.120, 2.210],
    [1.150, 2.210], [1.180, 2.210], [1.210, 2.210], [1.240, 2.210], [1.270, 2.210],
    # 第8行 (y=2.240)
    [1.000, 2.240], [1.030, 2.240], [1.060, 2.240], [1.090, 2.240], [1.120, 2.240],
    [1.150, 2.240], [1.180, 2.240], [1.210, 2.240], [1.240, 2.240], [1.270, 2.240],
    # 第9行 (y=2.270)
    [1.000, 2.270], [1.030, 2.270], [1.060, 2.270], [1.090, 2.270], [1.120, 2.270],
    [1.150, 2.270], [1.180, 2.270], [1.210, 2.270], [1.240, 2.270], [1.270, 2.270],
    # 第10行 (y=2.300)
    [1.000, 2.300], [1.030, 2.300], [1.060, 2.300], [1.090, 2.300], [1.120, 2.300],
    [1.150, 2.300], [1.180, 2.300], [1.210, 2.300], [1.240, 2.300], [1.270, 2.300],
    # 第11行 (y=2.330)
    [1.000, 2.330], [1.030, 2.330], [1.060, 2.330], [1.090, 2.330], [1.120, 2.330],
    [1.150, 2.330], [1.180, 2.330], [1.210, 2.330], [1.240, 2.330], [1.270, 2.330],
    # 第12行 (y=2.360)
    [1.000, 2.360], [1.030, 2.360], [1.060, 2.360], [1.090, 2.360], [1.120, 2.360],
    [1.150, 2.360], [1.180, 2.360], [1.210, 2.360], [1.240, 2.360], [1.270, 2.360],
    # 第13行 (y=2.390)
    [1.000, 2.390], [1.030, 2.390], [1.060, 2.390], [1.090, 2.390], [1.120, 2.390],
    [1.150, 2.390], [1.180, 2.390], [1.210, 2.390], [1.240, 2.390], [1.270, 2.390],
]

In [7]:
if __name__ == '__main__':
    print(get_put_table(cube_count, place_order))

极限填满行 12
([[9.0, 1.5, 0.0, 'ZL', 1], [7.0, 1.5, 0.0, 'ZL', 3], [5.0, 1.5, 0.0, 'ZL', 2], [4.5, 4.0, 90.0, 'ZL', 0], [2.5, 1.0, 0.0, 'line', 1], [6.5, 3.0, 0.0, 'line', 2], [9.0, 4.5, 90.0, 'line', 3], [10.0, 3.5, 90.0, 'line', 0], [7.0, 4.5, 0.0, 'T', 0], [2.0, 2.5, 0.0, 'T', 2], [2.5, 4.0, -90.0, 'T', 1], [6.0, 5.5, 180.0, 'T', 3], [8.0, 6.0, 90.0, 'LR', 0], [1.0, 4.0, 90.0, 'LR', 2], [2.0, 6.0, 0.0, 'LR', 1], [4.0, 6.0, 90.0, 'LR', 3], [3.0, 7.5, 0.0, 'ZR', 2], [2.0, 8.5, 0.0, 'ZR', 1], [4.5, 9.0, 90.0, 'ZR', 3], [7.0, 7.5, 0.0, 'ZR', 0], [2.0, 10.0, 180.0, 'LL', 2], [6.0, 9.0, -90.0, 'LL', 1], [7.0, 10.0, 90.0, 'LL', 3], [8.0, 11.0, -90.0, 'LL', 4], [10.0, 7.0, -90.0, 'LL', 0], [9.5, 9.5, 0.0, 'O', 2], [9.5, 11.5, 0.0, 'O', 1], [5.5, 11.5, 0.0, 'O', 3], [1.5, 11.5, 0.0, 'O', 0], [3.5, 11.5, 0.0, 'O', 4]], '12')
